# 020 - Area of applicability of the Carpathian extension
-------
Reports the results of a Meyer & Pebesma (2021) area-of-applicability (AOA) analysis of the deployed XGBoost `baseline_tessera` model over the **entire Carpathian mountain range**.

Run the scripts first:

```bash
screen -S carpathian_downloads
scripts/launch_carpathian_downloads.sh 100        # analysis grid resolution in metres

screen -S carp_aoa
.venv/bin/python -m scripts.run_area_of_applicability --grid-res 100
```
Key decisions:
1. **Analysis grid: 100 m** (point-sampled from the same 10 m feature space).
2. **Training points: the exact final-fit sample** (≤500 pixels per parcel, seed from
   `utils.terminology`), reproduced with `run_final_inference`'s sampling.
3. **Importance weights: the final model's own gain**, from a reproducible refit of
   its stored hyper-parameters (the deployed model was never serialised).
4. **Threshold: the paper's rule** — the outlier-removed maximum below the upper
   whisker (Q3 + 1.5·IQR) of the cross-validated training DIs; q95 reported as
   sensitivity only.
5. **CV structure: plain fold exclusion** over the six spatial blocks.
6. **Forest mask: WorldCover 2020 tree fraction** (covers Ukraine), applied at
   reporting time — masking cannot change DI values or the threshold.
7. **DI→performance: windowed PR-AUC** (primary) with windowed Brier as sensitivity,
   both with monotone fits.
8. **Feature space: the same TESSERA version on both sides**, the one notebook 001
   installed, re-sampled at the analysis resolution for the training points and the map.


In [ ]:
import json
import math
from pathlib import Path

import geopandas as gpd
import matplotlib.patheffects as path_effects
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch, Polygon, Rectangle
from matplotlib.ticker import FuncFormatter
from rasterio.vrt import WarpedVRT

from utils import raster_io, terminology
from utils.carpathians import (
    build_carpathian_countries,
    carpathian_countries_path,
    carpathian_grid,
    carpathians_massif_path,
    estimated_valid_pixels,
    load_carpathian_massif,
)
from utils.paths import get_project_paths
from utils.style import SEQUENTIAL_CMAP, save_figure, use_publication_style
from utils.vector_io import audit_vector, load_aoi

use_publication_style()
NOTEBOOK = "013_area_of_applicability"

paths = get_project_paths()
NODATA = terminology.NODATA

MASSIFS_SOURCE = paths.raw / "vectors" / "european_mountain_areas" / "m_massifs_v1.shp"
MASSIF_PATH = carpathians_massif_path(paths.repo_root)
CORINE_FOREST_PATH = (
    paths.processed / "vectors" / "corine_land_cover" / "corine_forest_carpathians_3035.gpkg"
)
COUNTRIES_PATH = carpathian_countries_path(paths.repo_root)
CARPATHIAN_RASTER_ROOT = paths.processed / "rasters" / "carpathians"
AOA_DIR = paths.results / "area_of_applicability" / "baseline_tessera"

TEAL = terminology.PALETTE_CATEGORICAL["teal"]
ORANGE = terminology.PALETTE_CATEGORICAL["orange"]
BLUE = terminology.PALETTE_CATEGORICAL["blue"]
MAGENTA = terminology.PALETTE_CATEGORICAL["magenta"]
LIGHT_GREEN = terminology.PALETTE_CATEGORICAL["light_green"]
YELLOW = terminology.PALETTE_CATEGORICAL["yellow"]
# Darker companion to the palette greens for the top expected-PR-AUC class.
DARK_GREEN = "#2E7D32"
# Non-forest grey matching the archive prototype reference figure.
ARCHIVE_GREY = "#e2e2e2"

print(f"Massif:        {MASSIF_PATH.relative_to(paths.repo_root)}")
print(f"CORINE forest: {CORINE_FOREST_PATH.relative_to(paths.repo_root)}")
print(f"Countries:     {COUNTRIES_PATH.relative_to(paths.repo_root)}")
print(f"AOA results:   {AOA_DIR.relative_to(paths.repo_root)}")

In [ ]:
# Map frame in the location-inset format (notebook 012) minus the tile background:
# Web Mercator km-formatted flush axes, a black frame, the Mercator-corrected scale
# bar in the bottom-right corner and a north arrow tight in the top-left corner.
# All 020 maps share this frame.
MAP_CRS = "EPSG:3857"


def carpathian_basemap(massif_3857, *, fig_height=4.6, margin_frac=0.035):
    """Figure and axes framed on the massif with the OpenTopoMap background."""
    bx0, by0, bx1, by1 = massif_3857.total_bounds
    pad_x = margin_frac * (bx1 - bx0)
    pad_y = margin_frac * (by1 - by0)
    x0, x1, y0, y1 = bx0 - pad_x, bx1 + pad_x, by0 - pad_y, by1 + pad_y
    fig, ax = plt.subplots(figsize=(fig_height * (x1 - x0) / (y1 - y0), fig_height), dpi=300)
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.set_aspect("equal", adjustable="box")
    return fig, ax, (x0, y0, x1, y1)


def finish_basemap(ax, extent, *, scale_km=100.0, tick_font=7.0, label_font=8.0, scale_font=6.0):
    """Axes, credit, scale bar and north arrow, as the 018 location inset."""
    x0, y0, x1, y1 = extent
    km_fmt = FuncFormatter(lambda v, _pos: f"{v / 1000:,.0f}")
    ax.xaxis.set_major_formatter(km_fmt)
    ax.yaxis.set_major_formatter(km_fmt)
    ax.tick_params(labelsize=tick_font, length=2.5, pad=1.5)
    ax.set_xlabel(f"Easting (km, {MAP_CRS})", fontsize=label_font, labelpad=2.0)
    ax.set_ylabel(f"Northing (km, {MAP_CRS})", fontsize=label_font, labelpad=2.0)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)
        spine.set_edgecolor("black")
    # Web Mercator inflates distance by 1/cos(latitude): the bar spans the map-unit
    # length of a true scale_km at the view's central latitude.
    centre_lat = math.degrees(2.0 * math.atan(math.exp(((y0 + y1) / 2) / 6378137.0)) - math.pi / 2)
    bar_len = scale_km * 1000.0 / math.cos(math.radians(centre_lat))
    bar_seg = bar_len / 4
    bar_h = 0.018 * (y1 - y0)
    bar_x = x1 - 0.015 * (x1 - x0) - bar_len
    bar_y = y0 + 0.012 * (y1 - y0)
    for k in range(4):
        ax.add_patch(
            Rectangle(
                (bar_x + k * bar_seg, bar_y),
                bar_seg,
                bar_h,
                facecolor="black" if k % 2 == 0 else "white",
                edgecolor="black",
                linewidth=0.4,
                zorder=7,
            )
        )
    ax.text(
        bar_x + bar_len / 2,
        bar_y + bar_h + 0.008 * (y1 - y0),
        f"{scale_km:.0f} km",
        ha="center",
        va="bottom",
        fontsize=scale_font,
        color="black",
        path_effects=[path_effects.withStroke(linewidth=1.4, foreground="white")],
        zorder=7,
    )
    size_n = 0.06 * min(x1 - x0, y1 - y0)
    cx_n, cy_n = x0 + 0.030 * (x1 - x0), y1 - 0.075 * (y1 - y0)
    half_n = 0.36 * size_n
    ax.add_patch(
        Polygon(
            [
                (cx_n, cy_n + 0.40 * size_n),
                (cx_n + half_n, cy_n - 0.40 * size_n),
                (cx_n, cy_n - 0.16 * size_n),
                (cx_n - half_n, cy_n - 0.40 * size_n),
            ],
            closed=True,
            facecolor="black",
            edgecolor="white",
            linewidth=0.5,
            clip_on=False,
            zorder=8,
        )
    )
    ax.text(
        cx_n,
        cy_n + 0.40 * size_n + 0.004 * (y1 - y0),
        "N",
        ha="center",
        va="bottom",
        fontsize=6,
        color="black",
        path_effects=[path_effects.withStroke(linewidth=1.4, foreground="white")],
        zorder=8,
    )


def raster_overlay_3857(path, *, band=1, max_px=2200):
    """Read a band warped to the map CRS, decimated, with its imshow extent."""
    with rasterio.open(path) as src:
        with WarpedVRT(src, crs=MAP_CRS) as vrt:
            decim = max(1, vrt.width // max_px)
            arr = vrt.read(band, out_shape=(vrt.height // decim, vrt.width // decim))
            b = vrt.bounds
    return arr, (b.left, b.right, b.bottom, b.top)


print("map-frame helpers ready (018 location-inset format, no tile background)")

## 1. Carpathian massif polygon

In [ ]:
massif = load_carpathian_massif(MASSIF_PATH)
print(f"[verify] {audit_vector(MASSIF_PATH)}")
area_attr = float(massif["area_km2"].iloc[0])
area_geom = float(massif.geometry.area.iloc[0]) / 1e6
print(f"[verify] area: attribute {area_attr:,.0f} km2, geometry {area_geom:,.0f} km2")
if abs(area_attr - area_geom) / area_attr > 0.02:
    raise RuntimeError("Geometry area deviates >2% from the source attribute; inspect.")
massif_3857 = massif.to_crs(MAP_CRS)
aoi = load_aoi(dissolve=True)

In [ ]:
# Map 1 - the massif, its nested sub-ranges and the Făgăraș training AOI.
sub_ranges = gpd.read_file(MASSIFS_SOURCE)
sub_ranges = sub_ranges[sub_ranges["m_massive"] == "carp"].to_crs(MAP_CRS)

fig, ax, extent = carpathian_basemap(massif_3857)
massif_3857.plot(ax=ax, facecolor="0.94", edgecolor=BLUE, linewidth=1.2, zorder=1)
sub_ranges.boundary.plot(ax=ax, color=BLUE, linewidth=0.2, alpha=0.45, zorder=2, rasterized=True)
aoi.to_crs(MAP_CRS).plot(ax=ax, facecolor="none", edgecolor=ORANGE, linewidth=1.4, zorder=4)
ax.legend(
    handles=[
        Patch(facecolor="none", edgecolor=BLUE, label="Carpathian massif"),
        Patch(facecolor="none", edgecolor=ORANGE, label="Făgăraș training AOI"),
    ],
    loc="lower left",
    fontsize=6,
)
finish_basemap(ax, extent)
save_figure(
    fig, f"{NOTEBOOK}/carpathian_massif_context", dpi=200, bbox_inches="tight", pad_inches=0.02
)
plt.show()

print(
    f"[verify] training AOI within massif: {aoi.geometry.iloc[0].within(massif.geometry.iloc[0])}"
)

## 2. CORINE CLC2018 forest layer

In [ ]:
if not CORINE_FOREST_PATH.exists():
    print("Not built yet - run: python -m scripts.download_carpathians_corine")
    forest = None
else:
    forest = gpd.read_file(CORINE_FOREST_PATH)
    print(f"[verify] {audit_vector(CORINE_FOREST_PATH)}")
    summary = (
        forest.assign(area_km2=forest.geometry.area / 1e6)
        .groupby(["country", "forest_type"])["area_km2"]
        .sum()
        .round(0)
        .unstack(fill_value=0)
    )
    print(summary.to_string())
    forest_km2 = forest.geometry.area.sum() / 1e6
    massif_km2 = massif.geometry.area.iloc[0] / 1e6
    print(f"[verify] forest {forest_km2:,.0f} km2 = {forest_km2 / massif_km2:.0%} of massif")

In [ ]:
# Map 2 - CORINE forest coverage; the Ukrainian CLC2018 gap shows as basemap-only
# massif interior.
if forest is not None:
    type_colours = {"broadleaf": TEAL, "coniferous": BLUE, "mixed": ORANGE}
    fig, ax, extent = carpathian_basemap(massif_3857)
    massif_3857.plot(ax=ax, facecolor="0.94", edgecolor=BLUE, linewidth=1.0, zorder=1)
    forest_3857 = forest.to_crs(MAP_CRS)
    for f_type, colour in type_colours.items():
        forest_3857[forest_3857["forest_type"] == f_type].plot(
            ax=ax, color=colour, linewidth=0, alpha=0.85, zorder=2, rasterized=True
        )
    aoi.to_crs(MAP_CRS).plot(ax=ax, facecolor="none", edgecolor="black", linewidth=1.2, zorder=4)
    ax.legend(
        handles=[Patch(color=c, label=t) for t, c in type_colours.items()]
        + [Patch(facecolor="none", edgecolor="black", label="training AOI")],
        loc="lower left",
        fontsize=6,
    )
    finish_basemap(ax, extent)
    save_figure(
        fig, f"{NOTEBOOK}/carpathian_corine_forest", dpi=200, bbox_inches="tight", pad_inches=0.02
    )
    plt.show()

## 3. Predictor raster and forest-mask verification

In [ ]:
def _discover(prefix: str) -> list[Path]:
    if not CARPATHIAN_RASTER_ROOT.exists():
        return []
    return sorted(CARPATHIAN_RASTER_ROOT.glob(f"{prefix}_*m"))


def _res_of(directory: Path, prefix: str) -> int:
    return int(directory.name.replace(f"{prefix}_", "").rstrip("m"))


def _tessera_mosaic(directory: Path, res: int) -> Path | None:
    tif = directory / f"tessera_2020_3035_{res}m.tif"
    vrt = directory / f"tessera_2020_3035_{res}m.vrt"
    return tif if tif.exists() else (vrt if vrt.exists() else None)


BASELINE_DIRS = _discover("baseline")
TESSERA_DIRS = _discover("tessera")
WORLDCOVER_DIRS = _discover("worldcover")
if not (BASELINE_DIRS or TESSERA_DIRS or WORLDCOVER_DIRS):
    print("No Carpathian rasters yet - run scripts/launch_carpathian_downloads.sh first.")
else:
    print(f"Baseline resolutions:   {[_res_of(d, 'baseline') for d in BASELINE_DIRS]}")
    print(f"TESSERA resolutions:    {[_res_of(d, 'tessera') for d in TESSERA_DIRS]}")
    print(f"WorldCover resolutions: {[_res_of(d, 'worldcover') for d in WORLDCOVER_DIRS]}")

In [ ]:
# Baseline: the four expected rasters per resolution, grid geometry, band names in
# canonical order, and per-band stats on a centre window.
BASELINE_FILES = ("elevation", "slope_deg", "heat_load_index", "distance_to_roads")

for directory in BASELINE_DIRS:
    res = _res_of(directory, "baseline")
    grid = carpathian_grid(massif, float(res))
    print(f"=== baseline @ {res} m ===")
    band_names = []
    ok = True
    for stem in BASELINE_FILES:
        path = directory / f"{stem}_3035_{res}m.tif"
        if not path.exists():
            print(f"  MISSING: {path.name} (script still running or step not done?)")
            ok = False
            continue
        audit = raster_io.audit_raster(path)
        print(f"  {audit}")
        with rasterio.open(path) as src:
            band_names.extend(src.descriptions)
            grid_ok = (
                src.crs == grid.crs
                and src.transform == grid.transform
                and (src.width, src.height) == (grid.width, grid.height)
            )
            if not grid_ok:
                print(f"  GRID MISMATCH vs carpathian_grid({res}) - investigate!")
                ok = False
            centre = rasterio.windows.Window(
                src.width // 2 - 512, src.height // 2 - 512, 1024, 1024
            )
            sample = src.read(window=centre)
            valid = sample[sample != NODATA]
            if valid.size:
                print(
                    f"    centre window: {valid.size:,}/{sample.size:,} valid, "
                    f"range {valid.min():.1f} to {valid.max():.1f}"
                )
    if ok and tuple(band_names) == terminology.FEATURE_SET_BANDS["baseline"]:
        print("  PASS: band names match FEATURE_SET_BANDS['baseline'] in order")
    elif ok:
        print(f"  BAND NAME MISMATCH: {band_names}")

In [ ]:
# TESSERA: the mosaic per resolution - 128 canonical bands, full-grid extent, tile
# bookkeeping from run_metadata, valid fraction vs the massif area.
for directory in TESSERA_DIRS:
    res = _res_of(directory, "tessera")
    grid = carpathian_grid(massif, float(res))
    print(f"=== tessera @ {res} m ===")
    meta_path = directory / "run_metadata.json"
    if meta_path.exists():
        meta = json.loads(meta_path.read_text())
        print(
            f"  run_metadata: {meta['tiles_written']} written, {meta['tiles_skipped']} skipped, "
            f"{len(meta['tiles_failed'])} failed, "
            f"{len(meta['mirror_missing_tiles'])} not on the mirror"
        )
        if meta["tiles_failed"]:
            print("  RERUN the script to fetch failed tiles before analysis.")
    n_tiles = (
        len(list((directory / "tiles").glob("*.tif"))) if (directory / "tiles").exists() else 0
    )
    print(f"  tiles on disk: {n_tiles}")
    mosaic = _tessera_mosaic(directory, res)
    if mosaic is None:
        print("  PENDING: mosaic (built after the tile loop completes)")
        continue
    print(f"  mosaic: {mosaic.name}")
    with rasterio.open(mosaic) as src:
        names = list(src.descriptions)
        expected_names = [
            b for b in terminology.FEATURE_SET_BANDS["baseline_tessera"] if b.startswith("tessera_")
        ]
        print(f"  {src.count} bands, {src.width}x{src.height} px, crs={src.crs}")
        grid_ok = src.transform == grid.transform and (src.width, src.height) == (
            grid.width,
            grid.height,
        )
        print(f"  {'PASS' if grid_ok else 'FAIL'}: mosaic grid matches carpathian_grid({res})")
        print(f"  {'PASS' if names == expected_names else 'FAIL'}: 128 canonical band names")
        decim = max(1, src.width // 2000)
        overview = src.read(1, out_shape=(src.height // decim, src.width // decim))
        valid_frac = float((overview != NODATA).mean())
        expected_frac = estimated_valid_pixels(massif, float(res)) / (grid.width * grid.height)
        print(
            f"  valid fraction (band 1, decimated): {valid_frac:.1%} "
            f"(massif/bbox expectation ~{expected_frac:.1%})"
        )

In [ ]:
# WorldCover tree fraction: the AOA forest mask. Fraction in [0, 1] inside the
# massif; "forest" at analysis time = fraction >= 0.5.
for directory in WORLDCOVER_DIRS:
    res = _res_of(directory, "worldcover")
    grid = carpathian_grid(massif, float(res))
    path = directory / f"worldcover_tree_fraction_3035_{res}m.tif"
    print(f"=== worldcover @ {res} m ===")
    if not path.exists():
        print(f"  MISSING: {path.name} (run scripts/download_carpathians_worldcover.py)")
        continue
    print(f"  {raster_io.audit_raster(path, with_stats=True)}")
    with rasterio.open(path) as src:
        grid_ok = src.transform == grid.transform and (src.width, src.height) == (
            grid.width,
            grid.height,
        )
        print(f"  {'PASS' if grid_ok else 'FAIL'}: grid matches carpathian_grid({res})")
        decim = max(1, src.width // 2000)
        overview = src.read(1, out_shape=(src.height // decim, src.width // decim))
        valid = overview[(overview != NODATA) & np.isfinite(overview)]
        in_range = bool(valid.size) and float(valid.min()) >= 0.0 and float(valid.max()) <= 1.0
        print(f"  {'PASS' if in_range else 'FAIL'}: fraction within [0, 1]")
        if valid.size:
            print(
                f"  forest-dominant share of massif (fraction >= 0.5): {(valid >= 0.5).mean():.1%}"
            )

In [ ]:
# Quicklook: three embedding dimensions as an RGB composite (diagnostic only; reads
# the consolidated mosaic through its overviews, so this is fast).
def _stretch(band: np.ndarray) -> np.ndarray:
    valid = np.isfinite(band) & (band != NODATA)
    if not valid.any():
        return np.zeros_like(band)
    lo, hi = np.percentile(band[valid], [2, 98])
    out = np.clip((band - lo) / (hi - lo + 1e-9), 0, 1)
    out[~valid] = 0
    return out


for directory in TESSERA_DIRS:
    res = _res_of(directory, "tessera")
    mosaic = _tessera_mosaic(directory, res)
    if mosaic is None:
        continue
    with rasterio.open(mosaic) as src:
        decim = max(1, src.width // 1500)
        shape = (src.height // decim, src.width // decim)
        rgb = np.stack([_stretch(src.read(b, out_shape=shape)) for b in (1, 33, 65)], axis=-1)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(rgb)
    ax.set_title(f"TESSERA 2020 @ {res} m - dims t000/t032/t064 as RGB (EPSG:3035 grid)")
    ax.set_axis_off()
    plt.show()

## 4. AOA results

In [ ]:
RESULTS_READY = (AOA_DIR / "run_metadata.json").exists()
if not RESULTS_READY:
    print("AOA results not present yet - run:")
    print("  .venv/bin/python -m scripts.run_area_of_applicability")
else:
    meta = json.loads((AOA_DIR / "run_metadata.json").read_text())
    thr = json.loads((AOA_DIR / "cv_di_threshold.json").read_text())
    RES = int(meta["grid_res_m"])
    print(
        f"Run: {meta['architecture']}__{meta['feature_set']} @ {RES} m "
        f"(training pixels: {meta['n_training']:,}, mapping run: {meta['mapping_run']})"
    )
    print(f"Feature space: {meta.get('feature_space', 'n/a')}")
    if "n_distinct_map_cells" in thr:
        print(
            f"Distinct map-side training vectors: {thr['n_distinct_map_cells']:,} "
            f"behind {thr['n']:,} training rows (duplicates share 100 m cells; they do "
            f"not change nearest-neighbour distances or the binary AOA)"
        )
    print(json.dumps(thr, indent=2))

In [ ]:
# Tables: threshold statistics, DI-denominator sensitivity, forest summary and the
# feature-weight breakdown (gain share of TESSERA vs baseline bands).
if RESULTS_READY:
    denom = pd.read_csv(AOA_DIR / "denominator_diagnostics.csv")
    print("DI denominator vs subset size (stability check):")
    print(denom.to_string(index=False))

    forest_summary = pd.read_csv(AOA_DIR / "forest_aoa_summary.csv")
    print("\nForest in/out of the AOA (WorldCover tree fraction >= 0.5):")
    print(forest_summary.round(1).to_string(index=False))

    weights = pd.read_csv(AOA_DIR / "feature_weights.csv")
    weights["group"] = np.where(
        weights["feature"].str.startswith("tessera_"), "TESSERA (128)", "baseline (6)"
    )
    share = weights.groupby("group")["importance"].sum()
    print("\nGain share by band group (linear, w):")
    print((100 * share / share.sum()).round(1).astype(str).add(" %").to_string())
    # Euclidean distance weights features by w^2, so the METRIC concentrates far
    # more than the linear shares suggest.
    metric_share = (weights["importance"] ** 2) / (weights["importance"] ** 2).sum()
    metric_group = metric_share.groupby(weights["group"]).sum()
    effective_dims = 1.0 / (metric_share**2).sum()
    print("\nMetric share by band group (w^2 - what the DI actually uses):")
    print((100 * metric_group).round(1).astype(str).add(" %").to_string())
    top_metric = metric_share.sort_values(ascending=False).head(5)
    print(
        "Top-5 metric shares: "
        + ", ".join(f"{weights['feature'][i]} {100 * v:.1f}%" for i, v in top_metric.items())
    )
    print(f"Effective dimensionality (participation ratio): {effective_dims:.1f} of 134")
    print("\nTop 15 features by gain:")
    print(weights.nlargest(15, "importance")[["feature", "importance"]].to_string(index=False))

    sens_path = AOA_DIR / "headline_sensitivity.csv"
    if sens_path.exists():
        print("\nHeadline sensitivity (threshold rule x forest-mask source):")
        print(pd.read_csv(sens_path).round(2).to_string(index=False))
        if "headline_slope_pp_per_001_di" in thr:
            print(
                f"Slope at the threshold: {thr['headline_slope_pp_per_001_di']:.1f} pp "
                f"of forest per 0.01 DI"
            )

In [ ]:
# Map 3 - the dissimilarity index over the basemap (sequential colour map).
if RESULTS_READY:
    di_img, di_extent = raster_overlay_3857(AOA_DIR / f"carpathian_di_3035_{RES}m.tif")
    di_img = np.where(np.isfinite(di_img) & (di_img != NODATA), di_img, np.nan)
    vmax = float(np.nanquantile(di_img, 0.99))
    fig, ax, extent = carpathian_basemap(massif_3857)
    shown = ax.imshow(
        di_img,
        cmap=SEQUENTIAL_CMAP,
        vmin=0,
        vmax=vmax,
        extent=di_extent,
        zorder=2,
        interpolation="nearest",
    )
    massif_3857.boundary.plot(ax=ax, color="black", linewidth=0.7, zorder=3)
    fig.colorbar(shown, ax=ax, shrink=0.65, label="Dissimilarity index (DI)")
    finish_basemap(ax, extent)
    save_figure(fig, f"{NOTEBOOK}/carpathian_di_map", dpi=200, bbox_inches="tight", pad_inches=0.02)
    plt.show()

In [ ]:
# Map 3b - the DI map cut at the AOA threshold: cells beyond the threshold are
# left white. The colour mapping is identical to Map 3 (vmin/vmax computed on
# the full DI distribution before cutting), so colours within the AOI do not
# change; the colorbar is cropped to the displayed range. AOI outlined black.
if RESULTS_READY:
    di_img, di_extent = raster_overlay_3857(AOA_DIR / f"carpathian_di_3035_{RES}m.tif")
    di_img = np.where(np.isfinite(di_img) & (di_img != NODATA), di_img, np.nan)
    vmax = float(np.nanquantile(di_img, 0.99))
    di_cut = np.where(di_img <= thr["aoa_threshold"], di_img, np.nan)
    fig, ax, extent = carpathian_basemap(massif_3857)
    shown = ax.imshow(
        di_cut,
        cmap=SEQUENTIAL_CMAP,
        vmin=0,
        vmax=vmax,
        extent=di_extent,
        zorder=2,
        interpolation="nearest",
    )
    massif_3857.boundary.plot(ax=ax, color="black", linewidth=0.7, zorder=3)
    aoi.to_crs(MAP_CRS).plot(ax=ax, facecolor="none", edgecolor="black", linewidth=1.0, zorder=4)
    cbar = fig.colorbar(shown, ax=ax, shrink=0.65, label="Dissimilarity index (DI)")
    cbar.ax.set_ylim(0, thr["aoa_threshold"])
    finish_basemap(ax, extent)
    save_figure(
        fig,
        f"{NOTEBOOK}/carpathian_di_map_thresholded",
        dpi=200,
        bbox_inches="tight",
        pad_inches=0.02,
    )
    plt.show()

In [ ]:
# Map 4 - the binary AOA (outside in the paper's magenta accent).
if RESULTS_READY:
    aoa_img, aoa_extent = raster_overlay_3857(AOA_DIR / f"carpathian_aoa_3035_{RES}m.tif")
    shown_img = np.full(aoa_img.shape, np.nan)
    shown_img[aoa_img == 1] = 0
    shown_img[aoa_img == 0] = 1
    inside_pct = float((aoa_img == 1).sum()) / max((aoa_img != 255).sum(), 1) * 100
    fig, ax, extent = carpathian_basemap(massif_3857)
    ax.imshow(
        shown_img,
        cmap=ListedColormap([BLUE, MAGENTA]),
        vmin=0,
        vmax=1,
        extent=aoa_extent,
        zorder=2,
        interpolation="nearest",
    )
    massif_3857.boundary.plot(ax=ax, color="black", linewidth=0.7, zorder=3)
    ax.legend(
        handles=[Patch(color=BLUE, label="Inside AOA"), Patch(color=MAGENTA, label="Outside AOA")],
        loc="lower left",
        fontsize=6,
    )
    finish_basemap(ax, extent)
    save_figure(
        fig, f"{NOTEBOOK}/carpathian_aoa_map", dpi=200, bbox_inches="tight", pad_inches=0.02
    )
    plt.show()

In [ ]:
# Map 5 - forest inside/outside the AOA (non-forest left to the basemap).
if RESULTS_READY:
    wc_path = (
        CARPATHIAN_RASTER_ROOT / f"worldcover_{RES}m" / f"worldcover_tree_fraction_3035_{RES}m.tif"
    )
    with rasterio.open(AOA_DIR / f"carpathian_aoa_3035_{RES}m.tif") as src:
        aoa_full = src.read(1)
        profile = {
            "transform": src.transform,
            "crs": src.crs,
            "width": src.width,
            "height": src.height,
        }
    with rasterio.open(wc_path) as src:
        frac_full = src.read(1)
    valid_full = aoa_full != 255
    forest_mask = valid_full & (frac_full != NODATA) & (frac_full >= 0.5)
    classes = np.zeros(aoa_full.shape, dtype=np.uint8)
    classes[valid_full & ~forest_mask] = 1
    classes[forest_mask & (aoa_full == 1)] = 2
    classes[forest_mask & (aoa_full == 0)] = 3

    from rasterio.io import MemoryFile

    with MemoryFile() as mem:
        with mem.open(driver="GTiff", count=1, dtype="uint8", nodata=0, **profile) as tmp:
            tmp.write(classes, 1)
        with mem.open() as tmp:
            with WarpedVRT(tmp, crs=MAP_CRS) as vrt:
                decim = max(1, vrt.width // 2200)
                cls_img = vrt.read(1, out_shape=(vrt.height // decim, vrt.width // decim))
                b = vrt.bounds

    shown_img = np.full(cls_img.shape, np.nan)
    shown_img[cls_img == 1] = 0
    shown_img[cls_img == 2] = 1
    shown_img[cls_img == 3] = 2
    n_forest = int((classes >= 2).sum())
    pct_in = 100 * float((classes == 2).sum()) / max(n_forest, 1)
    fig, ax, extent = carpathian_basemap(massif_3857)
    ax.imshow(
        shown_img,
        cmap=ListedColormap([ARCHIVE_GREY, TEAL, ORANGE]),
        vmin=0,
        vmax=2,
        extent=(b.left, b.right, b.bottom, b.top),
        zorder=2,
        interpolation="nearest",
    )
    massif_3857.boundary.plot(ax=ax, color="black", linewidth=0.7, zorder=3)
    aoi.to_crs(MAP_CRS).plot(ax=ax, facecolor="none", edgecolor="red", linewidth=0.7, zorder=4)
    mha = {c: float((classes == c).sum()) / 1e6 for c in (1, 2, 3)}  # 1 ha cells
    ax.legend(
        handles=[
            Patch(color=TEAL, label=f"Forest, inside AOA ({mha[2]:.1f} Mha)"),
            Patch(color=ORANGE, label=f"Forest, outside AOA ({mha[3]:.1f} Mha)"),
            Patch(color=ARCHIVE_GREY, label=f"Non-forest ({mha[1]:.1f} Mha)"),
            Patch(
                facecolor="none",
                edgecolor="red",
                label=f"Area of interest ({float(aoi.area.sum()) / 1e10:.1f} Mha)",
            ),
        ],
        loc="lower left",
        fontsize=8,
        frameon=False,
    )
    finish_basemap(ax, extent, scale_km=200.0, tick_font=9.0, label_font=10.0, scale_font=8.0)
    save_figure(
        fig,
        f"{NOTEBOOK}/fig_7_carpathian_forest_aoa_map",
        data=forest_summary,
        dpi=200,
        bbox_inches="tight",
        pad_inches=0.02,
    )
    plt.show()

In [ ]:
# Map 6 - expected PR-AUC: the isotonic DI-performance fit transferred onto the
# DI raster (a cell inherits the windowed OOF PR-AUC of training pixels at its
# DI). The fit is a step function - 11 plateaus over the training DI range - so
# the transferred surface is effectively categorical and is classed into four
# palette bands (teal = highest expected PR-AUC, then light green, yellow,
# orange). Outside-AOA cells are light grey: the curve is not extrapolated
# beyond the threshold. All valid massif cells are coloured, forest and
# non-forest alike (the transfer depends only on DI; forest masking is a
# separate, statistics-only step). Two panels: whole range + training AOI.
if RESULTS_READY:
    from rasterio.windows import bounds as window_bounds
    from rasterio.windows import from_bounds

    curve = pd.read_csv(AOA_DIR / "di_performance_curve.csv")
    fit = curve.dropna(subset=["pr_auc_fit"]).sort_values("di_mid")
    aoi_3857 = aoi.to_crs(MAP_CRS)

    def raster_window_3857(path, view, *, band=1, max_px=2200):
        with rasterio.open(path) as src, WarpedVRT(src, crs=MAP_CRS) as vrt:
            win = from_bounds(view[0], view[1], view[2], view[3], transform=vrt.transform)
            win = win.round_offsets().round_lengths()
            decim = max(1, int(win.width) // max_px)
            arr = vrt.read(
                band,
                window=win,
                out_shape=(max(int(win.height) // decim, 1), max(int(win.width) // decim, 1)),
            )
            wb = window_bounds(win, vrt.transform)
        return arr, (wb[0], wb[2], wb[1], wb[3])

    mx0, my0, mx1, my1 = massif_3857.total_bounds
    pad = 0.035
    view_a = (
        mx0 - pad * (mx1 - mx0),
        my0 - pad * (my1 - my0),
        mx1 + pad * (mx1 - mx0),
        my1 + pad * (my1 - my0),
    )
    bx0, by0, bx1, by1 = aoi_3857.total_bounds
    pad_b = 0.18
    view_b = (
        bx0 - pad_b * (bx1 - bx0),
        by0 - pad_b * (by1 - by0),
        bx1 + pad_b * (bx1 - bx0),
        by1 + pad_b * (by1 - by0),
    )

    di_path = AOA_DIR / f"carpathian_di_3035_{RES}m.tif"
    di_panels = []
    for view, max_px in ((view_a, 2400), (view_b, 1600)):
        di_win, ext = raster_window_3857(di_path, view, max_px=max_px)
        di_panels.append((di_win, ext, view))

    # No displayed cell falls below 0.60 in any of the three transfers (the
    # buffered fit bottoms out at ~0.62 inside the AOA threshold), so < 0.65
    # is the lowest band; every band above 0.65 uses a green.
    CLASS_EDGES = (0.65, 0.70, 0.80)
    CLASS_COLOURS = [YELLOW, LIGHT_GREEN, TEAL, DARK_GREEN]
    CLASS_LABELS = ["< 0.65", "0.65-0.70", "0.70-0.80", "> 0.80"]
    h_fig = 4.4
    w_a = h_fig * (view_a[2] - view_a[0]) / (view_a[3] - view_a[1])
    w_b = h_fig * (view_b[2] - view_b[0]) / (view_b[3] - view_b[1])

    def expected_class_map(expected_fn, save_name, legend_title, *, data=None):
        # expected_fn(di_values, panel_index, sel_mask) -> expected PR-AUC values
        # for the inside-AOA cells of that panel; classed with CLASS_EDGES.
        fig = plt.figure(figsize=(w_a + w_b + 1.2, h_fig), dpi=300)
        gs = fig.add_gridspec(1, 2, width_ratios=[w_a, w_b], wspace=0.16)
        for k, (di_win, ext, view) in enumerate(di_panels):
            ok = np.isfinite(di_win) & (di_win != NODATA)
            out_aoa = ok & (di_win > thr["aoa_threshold"])
            sel = ok & ~out_aoa
            exp = np.full(di_win.shape, np.nan)
            exp[sel] = expected_fn(di_win[sel], k, sel)
            exp_class = np.where(
                np.isfinite(exp), np.digitize(exp, CLASS_EDGES).astype(float), np.nan
            )
            outside = np.where(out_aoa, 1.0, np.nan)
            ax = fig.add_subplot(gs[0, k])
            ax.set_xlim(view[0], view[2])
            ax.set_ylim(view[1], view[3])
            ax.set_aspect("equal", adjustable="box")
            ax.imshow(
                exp_class,
                cmap=ListedColormap(CLASS_COLOURS),
                vmin=-0.5,
                vmax=3.5,
                extent=ext,
                zorder=2,
                interpolation="nearest",
            )
            ax.imshow(
                outside,
                cmap=ListedColormap([ARCHIVE_GREY]),
                vmin=0,
                vmax=1,
                extent=ext,
                zorder=2,
                interpolation="nearest",
            )
            massif_3857.boundary.plot(ax=ax, color="black", linewidth=0.7, zorder=3)
            aoi_3857.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=1.0, zorder=4)
            finish_basemap(ax, view, scale_km=100.0 if k == 0 else 20.0)
            if k == 1:
                ax.set_ylabel("")
        fig.axes[0].legend(
            handles=[
                *[
                    Patch(color=colour, label=label)
                    for colour, label in zip(CLASS_COLOURS[::-1], CLASS_LABELS[::-1], strict=True)
                ],
                Patch(color=ARCHIVE_GREY, label="Outside AOA"),
                Patch(facecolor="none", edgecolor="black", label="Training AOI"),
            ],
            title=legend_title,
            title_fontsize=7,
            loc="lower left",
            fontsize=6,
            frameon=False,
        )
        save_figure(
            fig, f"{NOTEBOOK}/{save_name}", data=data, dpi=200, bbox_inches="tight", pad_inches=0.02
        )
        plt.show()

    from matplotlib.colors import LinearSegmentedColormap

    GRADIENT_RAMP = LinearSegmentedColormap.from_list(
        "expected_ramp", [YELLOW, LIGHT_GREEN, TEAL, DARK_GREEN]
    )

    def expected_gradient_map(expected_fn, save_name, colorbar_label, *, data=None):
        # Continuous variant of the classed map: same panels and furniture, the
        # yellow -> light green -> teal -> dark green ramp bounded by the
        # displayed extremes, and a colorbar in place of the class legend.
        panel_exp = []
        for k, (di_win, ext, view) in enumerate(di_panels):
            ok = np.isfinite(di_win) & (di_win != NODATA)
            out_aoa = ok & (di_win > thr["aoa_threshold"])
            sel = ok & ~out_aoa
            exp = np.full(di_win.shape, np.nan)
            exp[sel] = expected_fn(di_win[sel], k, sel)
            panel_exp.append((exp, np.where(out_aoa, 1.0, np.nan), ext, view))
        vmin_g = float(np.nanmin([np.nanmin(e[0]) for e in panel_exp]))
        vmax_g = float(np.nanmax([np.nanmax(e[0]) for e in panel_exp]))
        print(f"{save_name}: displayed range {vmin_g:.3f} to {vmax_g:.3f}")
        fig = plt.figure(figsize=(w_a + w_b + 1.2, h_fig), dpi=300)
        gs = fig.add_gridspec(1, 2, width_ratios=[w_a, w_b], wspace=0.16)
        shown = None
        for k, (exp, outside, ext, view) in enumerate(panel_exp):
            ax = fig.add_subplot(gs[0, k])
            ax.set_xlim(view[0], view[2])
            ax.set_ylim(view[1], view[3])
            ax.set_aspect("equal", adjustable="box")
            shown = ax.imshow(
                exp,
                cmap=GRADIENT_RAMP,
                vmin=vmin_g,
                vmax=vmax_g,
                extent=ext,
                zorder=2,
                interpolation="nearest",
            )
            ax.imshow(
                outside,
                cmap=ListedColormap([ARCHIVE_GREY]),
                vmin=0,
                vmax=1,
                extent=ext,
                zorder=2,
                interpolation="nearest",
            )
            massif_3857.boundary.plot(ax=ax, color="black", linewidth=0.7, zorder=3)
            aoi_3857.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=1.0, zorder=4)
            finish_basemap(ax, view, scale_km=100.0 if k == 0 else 20.0)
            if k == 1:
                ax.set_ylabel("")
        fig.axes[0].legend(
            handles=[
                Patch(color=ARCHIVE_GREY, label="Outside AOA"),
                Patch(facecolor="none", edgecolor="black", label="Training AOI"),
            ],
            loc="lower left",
            fontsize=6,
            frameon=False,
        )
        fig.colorbar(shown, ax=fig.axes, shrink=0.72, pad=0.015, label=colorbar_label)
        save_figure(
            fig, f"{NOTEBOOK}/{save_name}", data=data, dpi=200, bbox_inches="tight", pad_inches=0.02
        )
        plt.show()

    expected_class_map(
        lambda d, k, sel: np.interp(d, fit["di_mid"], fit["pr_auc_fit"]),
        "expected_pr_auc_map",
        "Expected PR-AUC",
        data=curve,
    )

In [ ]:
# Map 6b - buffered expected PR-AUC: the transfer calibrated on the 10 km-
# buffered predictions (no training data within 10 km of the test parcels) -
# the regime that applies to almost all of the Carpathians. Nothing reaches
# 0.70 under this transfer (the fit spans ~0.62 at the AOA threshold to ~0.69
# at low DI), so the shared four-class view collapses; shown instead as a
# continuous ramp bounded by the lowest and highest displayed values.
if RESULTS_READY:
    curve_buf = pd.read_csv(AOA_DIR / "di_performance_curve_buffered.csv")
    fit_buf = curve_buf.dropna(subset=["pr_auc_fit"]).sort_values("di_mid")
    expected_gradient_map(
        lambda d, k, sel: np.interp(d, fit_buf["di_mid"], fit_buf["pr_auc_fit"]),
        "expected_pr_auc_map_buffered",
        "Expected PR-AUC (>= 10 km buffered)",
        data=curve_buf,
    )

In [ ]:
# Figure: DI-performance - windowed PR-AUC (primary, with lift over prevalence),
# ROC-AUC, F1 at each fold's inner-CV pixel threshold, and Brier (sensitivity),
# each with its monotone (isotonic) fit, from the pooled out-of-fold predictions.
# AUC-style metrics are set-level quantities - a single sample has no PR-AUC -
# so performance conditional on DI is estimated over sliding windows of
# neighbouring-DI samples (Brier alone would admit a per-sample plot).
if RESULTS_READY:
    curve = pd.read_csv(AOA_DIR / "di_performance_curve.csv")
    panels = [
        ("pr_auc", "PR-AUC", "window PR-AUC"),
        ("roc_auc", "ROC-AUC", "window ROC-AUC"),
        ("f1", "F1 (inner-fold threshold)", "window F1"),
        ("brier", "Brier score", "window Brier"),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
    for ax, (col, label, raw_label) in zip(axes.ravel(), panels, strict=True):
        ax.scatter(curve["di_mid"], curve[col], s=12, color=BLUE, alpha=0.6, label=raw_label)
        ax.plot(
            curve["di_mid"],
            curve[f"{col}_fit"],
            color=MAGENTA,
            linewidth=2,
            label="monotone-constrained fit",
        )
        if col == "pr_auc":
            ax.scatter(
                curve["di_mid"],
                curve["pr_auc_lift"],
                s=8,
                color=TEAL,
                alpha=0.5,
                label="lift over prevalence",
            )
        ax.axvline(
            thr["aoa_threshold"],
            color=ORANGE,
            linestyle="--",
            linewidth=1.2,
            label=f"AOA threshold = {thr['aoa_threshold']:.3f}",
        )
        ax.set_ylabel(label)
        ax.legend(fontsize=8)
    for ax in axes[1]:
        ax.set_xlabel("Cross-validated DI (window median)")
    fig.suptitle(
        "Out-of-fold performance vs DI (fits impose monotonicity; the observed decline "
        "concentrates above DI ~0.42)",
        fontsize=10,
    )
    save_figure(fig, f"{NOTEBOOK}/di_performance_curve", data=curve)
    plt.show()

In [ ]:
# Table: set-level performance in non-overlapping DI bins. Unlike the sliding
# windows above (which share samples), the bins partition the training pixels,
# so each row is an independent estimate. F1 classifies each sample at its own
# fold's inner-CV pixel threshold - the leakage-free operating point.
if RESULTS_READY:
    di_table = pd.read_csv(AOA_DIR / "di_performance_table.csv")
    shown_t = di_table.drop(columns=["di_lo", "di_hi"]).copy()
    shown_t["n"] = shown_t["n"].map("{:,}".format)
    print(shown_t.to_string(index=False, float_format=lambda v: f"{v:0.3f}"))
    above = di_table["di_lo"] >= thr["aoa_threshold"] - 1e-9
    print(
        f"\nTraining pixels with DI above the AOA threshold: "
        f"{int(di_table.loc[above, 'n'].sum()):,} of {int(di_table['n'].sum()):,}"
    )

### Buffered transfer: performance when training data is at least 10 km away

In [ ]:
# Buffered DI-performance table and the distance decay behind the blended map.
if RESULTS_READY:
    buf_table = pd.read_csv(AOA_DIR / "di_performance_table_buffered.csv")
    shown_b = buf_table.drop(columns=["di_lo", "di_hi"]).copy()
    shown_b["n"] = shown_b["n"].map("{:,}".format)
    print("DI-binned performance, 10 km-buffered predictions:")
    print(shown_b.to_string(index=False, float_format=lambda v: f"{v:0.3f}"))

    decay = pd.read_csv(AOA_DIR / "distance_decay.csv")
    curve_buf = pd.read_csv(AOA_DIR / "di_performance_curve_buffered.csv")
    fit_buf = curve_buf.dropna(subset=["pr_auc_fit"]).sort_values("di_mid")
    fit0 = curve.dropna(subset=["pr_auc_fit"]).sort_values("di_mid")
    a_0 = float(decay["pr_auc_adjusted_fit"].iloc[0])
    a_far = float(decay["pr_auc_adjusted_fit"].iloc[-1])

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    axes[0].scatter(
        curve["di_mid"],
        curve["pr_auc"],
        s=8,
        color=BLUE,
        alpha=0.35,
        label="windows, fold-exclusion CV",
    )
    axes[0].plot(
        fit0["di_mid"], fit0["pr_auc_fit"], color=BLUE, linewidth=2, label="fit, fold-exclusion CV"
    )
    axes[0].scatter(
        curve_buf["di_mid"],
        curve_buf["pr_auc"],
        s=8,
        color=ORANGE,
        alpha=0.35,
        label="windows, 10 km buffered",
    )
    axes[0].plot(
        fit_buf["di_mid"],
        fit_buf["pr_auc_fit"],
        color=ORANGE,
        linewidth=2,
        label="fit, 10 km buffered",
    )
    axes[0].axvline(
        thr["aoa_threshold"],
        color="black",
        linestyle="--",
        linewidth=1,
        label=f"AOA threshold = {thr['aoa_threshold']:.3f}",
    )
    axes[0].set_xlabel("Cross-validated DI (window median)")
    axes[0].set_ylabel("PR-AUC")
    axes[0].set_title("The two DI transfers", fontsize=9)
    axes[0].legend(fontsize=7)
    axes[1].plot(
        decay["gap_km"],
        decay["pr_auc_buffered"],
        "o-",
        color=ORANGE,
        markersize=3,
        linewidth=1,
        label="buffered arm",
    )
    axes[1].plot(
        decay["gap_km"],
        decay["pr_auc_control"],
        "o-",
        color=BLUE,
        markersize=3,
        linewidth=1,
        label="control arm (capacity only)",
    )
    axes[1].plot(
        decay["gap_km"],
        decay["pr_auc_proximity_adjusted"],
        "o",
        color=TEAL,
        markersize=4,
        label="proximity-only (buffered - capacity)",
    )
    axes[1].plot(
        decay["gap_km"],
        decay["pr_auc_adjusted_fit"],
        color=TEAL,
        linewidth=2,
        label="isotonic fit (drives the blend weight)",
    )
    axes[1].set_xlabel("Gap between training and test data (km)")
    axes[1].set_ylabel("Pooled PR-AUC")
    axes[1].set_title(
        f"Distance decay: {a_0:.3f} at 0 km to {a_far:.3f}, clamped beyond "
        f"{int(decay['gap_km'].max())} km",
        fontsize=9,
    )
    axes[1].legend(fontsize=7)
    save_figure(fig, f"{NOTEBOOK}/buffered_transfer_and_decay", data=decay)
    plt.show()

## 5. Key statistics

In [ ]:
# S0 - assemble the reporting masks and lookups once.
if RESULTS_READY:
    RAW_DIR = paths.figures / NOTEBOOK / "raw"
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    THR = float(thr["aoa_threshold"])

    with rasterio.open(AOA_DIR / f"carpathian_di_3035_{RES}m.tif") as src:
        di_full = src.read(1)
        grid_t = src.transform
    with rasterio.open(AOA_DIR / f"carpathian_aoa_3035_{RES}m.tif") as src:
        aoa_full = src.read(1)
    with rasterio.open(AOA_DIR / f"carpathian_nearest_label_3035_{RES}m.tif") as src:
        nearest_full = src.read(1)
    with rasterio.open(
        CARPATHIAN_RASTER_ROOT / f"worldcover_{RES}m" / f"worldcover_tree_fraction_3035_{RES}m.tif"
    ) as src:
        frac_full = src.read(1)
    with rasterio.open(
        CARPATHIAN_RASTER_ROOT / f"baseline_{RES}m" / f"elevation_3035_{RES}m.tif"
    ) as src:
        elev_full = src.read(1)

    grid = carpathian_grid(massif, float(RES))
    valid_full = aoa_full != 255
    forest_full = valid_full & (frac_full != NODATA) & (frac_full >= 0.5)
    inside_full = aoa_full == 1
    CELL_MHA = (RES * RES) / 1e10  # one cell in Mha

    if not COUNTRIES_PATH.exists():
        # National boundaries for the per-country breakdown, assembled once from the
        # Geofabrik extracts of the baseline download (utils.carpathians).
        build_carpathian_countries(paths, massif_path=MASSIF_PATH)
    countries = gpd.read_file(COUNTRIES_PATH)
    country_codes = list(countries["country"])
    country_raster = raster_io.rasterize_mask(
        countries.geometry, grid, all_touched=False, fill=0, value=0, dtype="uint8"
    )
    country_raster = np.zeros(grid.shape, dtype=np.uint8)
    for i, geom in enumerate(countries.geometry, start=1):
        burned = raster_io.rasterize_mask([geom], grid, all_touched=False)
        country_raster[burned == 1] = i

    aoi_mask = raster_io.rasterize_mask(aoi.geometry, grid, all_touched=True) == 1

    if forest is not None:
        clc_raster = np.zeros(grid.shape, dtype=np.uint16)
        for clc_code in (311, 312, 313):
            geoms = forest.loc[forest["clc_code"] == clc_code, "geometry"]
            if len(geoms):
                burned = raster_io.rasterize_mask(geoms, grid, all_touched=False)
                clc_raster[burned == 1] = clc_code
    else:
        clc_raster = None
    print("reporting masks ready:", f"{valid_full.sum():,} valid cells")

In [ ]:
# S1 - three-class breakdown with nearest-analogue splits: region, countries, AOI.
if RESULTS_READY:

    def three_class_rows(region: str, mask: np.ndarray) -> list[dict]:
        v = valid_full & mask
        f = forest_full & mask
        land = float(v.sum())
        forest_n = float(f.sum())
        rows = []
        for label, cls in (
            ("forest inside AOA", f & inside_full),
            ("forest outside AOA", f & ~inside_full),
            ("non-forest", v & ~f),
        ):
            n = float(cls.sum())
            near_ogf = float((cls & (nearest_full == 1)).sum())
            rows.append(
                {
                    "region": region,
                    "class": label,
                    "area_Mha": n * CELL_MHA,
                    "pct_of_forest": 100 * n / forest_n
                    if label.startswith("forest") and forest_n
                    else np.nan,
                    "pct_of_land": 100 * n / land if land else np.nan,
                    "nearest_ogf_Mha": near_ogf * CELL_MHA,
                    "pct_nearest_ogf": 100 * near_ogf / n if n else np.nan,
                    "nearest_non_ogf_Mha": (n - near_ogf) * CELL_MHA,
                }
            )
        return rows

    all_rows = three_class_rows("Carpathians (all)", np.ones_like(valid_full, dtype=bool))
    for i, code_ in enumerate(country_codes, start=1):
        all_rows += three_class_rows(code_, country_raster == i)
    all_rows += three_class_rows("Făgăraș AOI", aoi_mask)
    three_class = pd.DataFrame(all_rows)
    region_cells = three_class.groupby("region")["area_Mha"].transform("sum") * 1e4  # kha -> cells
    three_class["reliable"] = region_cells * 100 >= 1_000  # >= 1,000 cells in the region
    pd.set_option("display.width", 220)
    print(three_class.round(3).to_string(index=False))
    small = three_class.loc[~three_class["reliable"], "region"].unique()
    if len(small):
        print(
            f"\nNOTE: regions below the 1,000-cell reporting threshold "
            f"(interpret with caution): {list(small)}"
        )
    three_class.to_csv(RAW_DIR / "paper_stats_three_class.csv", index=False)

In [ ]:
# S2 - stratification by altitude and by CORINE forest type (EU-side only).
if RESULTS_READY:
    bands = [
        (0, 400),
        (400, 700),
        (700, 1000),
        (1000, 1300),
        (1300, 1600),
        (1600, 2000),
        (2000, 2700),
    ]
    rows = []
    for lo, hi in bands:
        m = forest_full & (elev_full != NODATA) & (elev_full >= lo) & (elev_full < hi)
        n = float(m.sum())
        if n < 100:
            continue
        rows.append(
            {
                "altitude_band_m": f"{lo}-{hi}",
                "forest_Mha": n * CELL_MHA,
                "pct_forest_inside_aoa": 100 * float((m & inside_full).sum()) / n,
                "median_di": float(np.median(di_full[m])),
            }
        )
    altitude = pd.DataFrame(rows)
    print("Forest AOA by altitude band:")
    print(altitude.round(3).to_string(index=False))
    altitude.to_csv(RAW_DIR / "paper_stats_altitude.csv", index=False)

    if clc_raster is not None:
        rows = []
        for clc_code, name in terminology.CORINE_FOREST_CLASSES.items():
            m = valid_full & (clc_raster == clc_code)
            n = float(m.sum())
            if n < 100:
                continue
            rows.append(
                {
                    "forest_type": name,
                    "clc_code": clc_code,
                    "area_Mha": n * CELL_MHA,
                    "pct_inside_aoa": 100 * float((m & inside_full).sum()) / n,
                    "median_di": float(np.median(di_full[m])),
                }
            )
        clc_types = pd.DataFrame(rows)
        print("\nAOA by CORINE forest type (CLC countries only; Ukraine absent from CLC):")
        print(clc_types.round(3).to_string(index=False))
        clc_types.to_csv(RAW_DIR / "paper_stats_forest_type.csv", index=False)

        # The type split is confounded with elevation (broadleaf sits lower);
        # cross-tabulate pct-inside by type within elevation bands.
        xtab_rows = []
        for lo, hi in [(0, 700), (700, 1000), (1000, 1300), (1300, 2000)]:
            for clc_code, name in terminology.CORINE_FOREST_CLASSES.items():
                m = (
                    valid_full
                    & (clc_raster == clc_code)
                    & (elev_full != NODATA)
                    & (elev_full >= lo)
                    & (elev_full < hi)
                )
                n = float(m.sum())
                if n >= 1_000:
                    xtab_rows.append(
                        {
                            "altitude_band_m": f"{lo}-{hi}",
                            "forest_type": name,
                            "kha": n / 100,
                            "pct_inside_aoa": 100 * float((m & inside_full).sum()) / n,
                        }
                    )
        xtab = pd.DataFrame(xtab_rows).pivot_table(
            index="altitude_band_m", columns="forest_type", values="pct_inside_aoa"
        )
        print("\npct inside AOA, forest type x elevation (confound check):")
        print(xtab.round(1).to_string())
        xtab.to_csv(RAW_DIR / "paper_stats_type_by_elevation.csv")

In [ ]:
# S3 - within-AOI sense check against the published prediction map.
if RESULTS_READY:
    aoi_valid = valid_full & aoi_mask
    print(f"AOI cells on the Carpathian grid: {int(aoi_valid.sum()):,}")
    aoi_n = float(aoi_valid.sum())
    print(f"AOI inside AOA: {100 * float((aoi_valid & inside_full).sum()) / aoi_n:.1f}%")
    print(f"AOI median DI: {float(np.median(di_full[aoi_valid])):.3f} (threshold {THR:.3f})")
    aoi_near_ogf = 100 * float((aoi_valid & (nearest_full == 1)).sum()) / aoi_n
    print(f"AOI nearest-analogue = OGF: {aoi_near_ogf:.1f}%")

    binary_path = paths.results / "final" / "ogf_binary_3035_10m.tif"
    if binary_path.exists():
        with rasterio.open(binary_path) as src:
            pred = src.read(1)
            pt = src.transform
        rows_g, cols_g = np.where(aoi_valid)
        xs = grid_t.c + (cols_g + 0.5) * grid_t.a
        ys = grid_t.f + (rows_g + 0.5) * grid_t.e
        pc = np.floor((xs - pt.c) / pt.a).astype(int)
        pr = np.floor((ys - pt.f) / pt.e).astype(int)
        ok = (pc >= 0) & (pc < pred.shape[1]) & (pr >= 0) & (pr < pred.shape[0])
        sampled = np.full(len(rows_g), 255, dtype=pred.dtype)
        sampled[ok] = pred[pr[ok], pc[ok]]
        pred_valid = np.isin(sampled, (0, 1))
        pred_ogf = sampled == 1
        near_ogf = nearest_full[aoi_valid][pred_valid] == 1
        print(f"\nPublished-map cells sampled at AOI cell centres: {int(pred_valid.sum()):,}")
        print(f"predicted-OGF share (published map): {100 * pred_ogf[pred_valid].mean():.1f}%")
        print(f"nearest-analogue-OGF share (AOA run): {100 * near_ogf.mean():.1f}%")
        print(
            "(Different constructs - a prediction vs the label of the nearest training "
            "analogue - compared only as an order-of-magnitude sense check, not as an "
            "accuracy measure.)"
        )
    else:
        print("published binary map not found; skipped prediction comparison")